# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook walks through loading and exploring the [FAIR\u005e2](https://doi.org/10.71728/senscience.vq4a-28xa) protein mass spectrometry dataset using the `mlcroissant` library, referencing all entities by their `@id`.

### Dataset Source
The dataset source is referenced by its Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")
print(f"\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")


## 2. Data Overview
List all available record sets, their fields, and corresponding `@id` identifiers.

This step ensures you know which tables (record sets) exist and which fields they expose, referencing each strictly by `@id`.

In [ ]:
# Gather record sets and their fields by @id
record_sets = getattr(dataset.metadata, 'recordSet', [])

if not record_sets:
    print("No record sets were found in this dataset metadata.")
else:
    for record_set_obj in record_sets:
        # Each record_set_obj is an mlc.RecordSet, get its @id and fields
        rs_id = getattr(record_set_obj, '@id', None)
        print(f"RecordSet @id: {rs_id}")
        if hasattr(record_set_obj, 'field'):
            fields = record_set_obj.field
            for field in fields:
                field_id = getattr(field, '@id', None)
                label = getattr(field, 'label', getattr(field, 'name', None))
                print(f"  Field @id: {field_id}\tLabel: {label}")
        print()
    print("You should select the appropriate record set(s) and field @ids for the following analyses.")

## 3. Data Extraction
Load all data from a record set into a DataFrame.

You must reference record sets and fields strictly by their `@id`. Replace `<record_set_id>` below with the actual `@id` as shown in the overview.

In [ ]:
# Example record set listing (run previous cell to view exact IDs)
# For this dataset, we'll dynamically list all record sets (even though most Croissant datasets only contain one main table)

# Find all available record set @ids
record_sets = getattr(dataset.metadata, 'recordSet', [])
record_set_ids = []
for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    if rs_id is not None:
        record_set_ids.append(rs_id)

dataframes = {}
if not record_set_ids:
    print("No concrete record sets with @id detected in the Croissant metadata.")
else:
    for record_set_id in record_set_ids:
        print(f"Loading records from RecordSet {record_set_id} ...")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"No records found for {record_set_id}.")

# For further analysis, select the main data record set's @id. If only one, you can do:

if record_set_ids:
    main_record_set_id = record_set_ids[0]


## 4. Exploratory Data Analysis (EDA)
Demonstrate common data manipulations using `@id` references for all entities.
We'll select a numeric field, filter, normalize, and group the data for analysis.

In [ ]:
import numpy as np

# Please check previous cells for correct DataFrame keys and field/column names.
# Let's auto-select a numeric field; users may override with the appropriate field @id as needed.

if dataframes:
    df = dataframes[main_record_set_id]
    # Try to find a numeric column
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        # Try to cast numeric-looking fields
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c], errors='ignore')
            except Exception:
                pass
        numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()

    if not numeric_candidates:
        print("No numeric fields found. Please review column headers and select a suitable field with a numeric type.")
    else:
        numeric_field = numeric_candidates[0]  # Use the first detected numeric field
        print(f"Proceeding with numeric field: {numeric_field}")

        # Filtering by a threshold (demonstration: pick the mean as a threshold)
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.3f}")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Group by another field if available (try to find a string/categorical candidate excluding numeric_field)
        other_fields = [c for c in df.columns if c != numeric_field]
        group_field = None
        for c in other_fields:
            if df[c].dtype == object and df[c].nunique() < 20:
                group_field = c
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped (mean) {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical grouping field found.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between record set fields, referencing them by `@id` (shown as column names here).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    # Example: histogram of the selected numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} (field @id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    # If grouped data exists, plot group means
    if 'grouped_df' in locals() and group_field is not None:
        plt.figure(figsize=(8, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated:
- How to load FAIR\u005e2 mass spectrometry dataset metadata and records with `mlcroissant` using strict `@id` referencing for all entities.
- How to extract, filter, normalize, and group data using field and record set `@id`s as references.
- How to conduct basic visualization of data distributions and relationships.

For further FAIR, reproducible science, always reference Croissant entities by `@id` and consult the [Croissant specification](https://mlcommons.org/croissant/).